In [18]:
import pandas as pd

csv_path = '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/19_Lucia/03_C_scan/panels_procedures_v02.csv'
panels_df = pd.read_csv(csv_path)
panels_df

,Sample name,Description,Stacking sequence,Laying up method,Curing process,Characterization methods
0,Panel 1,Cured by Hexcel (last presentation),[+/0/-/90]2s,HLU,Vacuum bag,"C-scan, XCT, DSC"
1,Panel 2,Laid up by Hexcel/ Cured at Imdea,[+/0/-/90]3s,HLU,Vacuum bag,"C-scan, XCT, DSC"
2,Panel 3,Laid up by Hexcel/ Cured at Imdea,[+/0/-/90]3s,HLU,Hot press (7bar),"C-scan (cured), XCT (uncured)"
3,Panel 4,Laid up by Hexcel/ Cured at Imdea,[+/0/-/90]3s,HLU,Vacuum bag,"C-scan, XCT"
4,Panel 5,Laid up by Hexcel/,[+/0/-/90]3s,HLU,Vacuum bag (remove film),C-scan
5,Panel 6,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t1),"C-scan, XCT, DSC"
6,Panel 7,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t2),"C-scan, XCT, DSC"
7,Panel 8,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t3),"C-scan, XCT, DSC"
8,Panel 9,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t4),"C-scan, XCT, DSC"
9,Panel 10,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t4),"C-scan, XCT, DSC"


In [19]:
unique_combinations = panels_df[['Stacking sequence', 'Laying up method', 'Curing process']].drop_duplicates().reset_index(drop=True)
unique_combinations

,Stacking sequence,Laying up method,Curing process
0,[+/0/-/90]2s,HLU,Vacuum bag
1,[+/0/-/90]3s,HLU,Vacuum bag
2,[+/0/-/90]3s,HLU,Hot press (7bar)
3,[+/0/-/90]3s,HLU,Vacuum bag (remove film)
4,[+/0/-/90]3s,HLU,Vacuum bag (t1)
5,[+/0/-/90]3s,HLU,Vacuum bag (t2)
6,[+/0/-/90]3s,HLU,Vacuum bag (t3)
7,[+/0/-/90]3s,HLU,Vacuum bag (t4)
8,[+/0/-/90]3s,HLU,Vacuum bag (full vacuum maintained)
9,[+/0/-/90]3s,AFP,Uncured


In [20]:
import sys
sys.path.append('../')
import dbtools.dbtools as qrs
import dbtools.load as load
import psycopg2

try:
    conn = qrs.connect()
    print("Connected to the database")
except (Exception, psycopg2.DatabaseError) as error:
    print(error)

Connected to the database


In [21]:
existing_fabrications = qrs.get_data('fabrications')
existing_names = set(existing_fabrications['name_fabrication'])

fabrication_ids = []

for idx, row in unique_combinations.iterrows():
    laying_up = row['Laying up method']
    curing = row['Curing process']
    stacking = row['Stacking sequence']

    name = f"AROA_{idx + 1:02d}_{laying_up}_{curing}".replace(' ', '_')

    if name in existing_names:
        fabrication_id = int(existing_fabrications.loc[existing_fabrications['name_fabrication'] == name, 'id_fabrication'].iloc[0])
        print(f"Fabrication '{name}' already exists with ID: {fabrication_id}")
    else:
        additional_metadata = [
            {'key': 'laying_up_method', 'value': laying_up, 'type': 'nominal'},
            {'key': 'curing_process', 'value': curing, 'type': 'text'},
        ]
        if pd.notna(stacking):
            additional_metadata.append({'key': 'stacking_sequence', 'value': stacking, 'type': 'nominal'})

        fabrication_id = load.load_fabrication(conn, name, additional_metadata)
        existing_names.add(name)

    fabrication_ids.append(fabrication_id)

unique_combinations['fabrication_id'] = fabrication_ids
unique_combinations

Fabrication 'AROA_01_HLU_Vacuum_bag' already exists with ID: 15
Fabrication 'AROA_02_HLU_Vacuum_bag' already exists with ID: 16
Fabrication 'AROA_03_HLU_Hot_press__(7bar)' already exists with ID: 17
Fabrication 'AROA_04_HLU_Vacuum_bag_(remove_film)' already exists with ID: 18
Fabrication 'AROA_05_HLU_Vacuum_bag_(t1)' already exists with ID: 19
Fabrication 'AROA_06_HLU_Vacuum_bag_(t2)' already exists with ID: 20
Fabrication 'AROA_07_HLU_Vacuum_bag_(t3)' already exists with ID: 21
Fabrication 'AROA_08_HLU_Vacuum_bag_(t4)' already exists with ID: 22
Fabrication 'AROA_09_HLU_Vacuum_bag_(full_vacuum_maintained)' already exists with ID: 23
Fabrication 'AROA_10_AFP_Uncured' already exists with ID: 24
Fabrication 'AROA_11_AFP_Vacuum_bag' already exists with ID: 25
Fabrication 'AROA_12_AFP_Vacuum_bag' already exists with ID: 26
Fabrication 'AROA_13_AFP_Uncured' already exists with ID: 27
Fabrication 'AROA_14_HLU_Vacuum_bag_and_dwell_+_30min.' already exists with ID: 28
Fabrication 'AROA_15_HLU_

,Stacking sequence,Laying up method,Curing process,fabrication_id
0,[+/0/-/90]2s,HLU,Vacuum bag,15
1,[+/0/-/90]3s,HLU,Vacuum bag,16
2,[+/0/-/90]3s,HLU,Hot press (7bar),17
3,[+/0/-/90]3s,HLU,Vacuum bag (remove film),18
4,[+/0/-/90]3s,HLU,Vacuum bag (t1),19
5,[+/0/-/90]3s,HLU,Vacuum bag (t2),20
6,[+/0/-/90]3s,HLU,Vacuum bag (t3),21
7,[+/0/-/90]3s,HLU,Vacuum bag (t4),22
8,[+/0/-/90]3s,HLU,Vacuum bag (full vacuum maintained),23
9,[+/0/-/90]3s,AFP,Uncured,24


In [22]:
materials = qrs.get_data('materials')
materials

,id_material,name_material
0,15,Fibra IM7 Resina M56
1,16,Material Hexcel 0.508 CPT
2,20,Placeholder Material


In [23]:
# Set the material_id to use when loading the panels below
material_id = 15

In [24]:
existing_panels = qrs.get_data('panels')
existing_panel_names = set(existing_panels['name_panel'])

def combo_key(row):
    return (
        row['Stacking sequence'] if pd.notna(row['Stacking sequence']) else None,
        row['Laying up method'],
        row['Curing process'],
    )

fabrication_lookup = {combo_key(row): row['fabrication_id'] for _, row in unique_combinations.iterrows()}

# Placeholder dimensions until real measurements are available
height = width = thickness = 1.0

panel_ids = []

for _, row in panels_df.iterrows():
    name = row['Sample name']

    if name in existing_panel_names:
        panel_id = int(existing_panels.loc[existing_panels['name_panel'] == name, 'id_panel'].iloc[0])
        print(f"Panel '{name}' already exists with ID: {panel_id}")
        panel_ids.append(panel_id)
        continue

    fabrication_id = int(fabrication_lookup[combo_key(row)])
    description = row['Description'] if pd.notna(row['Description']) else None

    additional_metadata = []
    if pd.notna(row['Stacking sequence']):
        additional_metadata.append({'key': 'stacking_sequence', 'value': row['Stacking sequence'], 'type': 'nominal'})
    if pd.notna(row['Characterization methods']):
        additional_metadata.append({'key': 'characterization_methods', 'value': row['Characterization methods'], 'type': 'text'})

    panel_id = load.load_panel(
        conn, name, material_id, fabrication_id, height, width, thickness,
        description=description, additional_metadata=additional_metadata
    )
    existing_panel_names.add(name)
    panel_ids.append(panel_id)

panels_df['panel_id'] = panel_ids
panels_df

Panel 'Panel 1' loaded with ID: 16
Panel 'Panel 2' loaded with ID: 17
Panel 'Panel 3' loaded with ID: 18
Panel 'Panel 4' loaded with ID: 19
Panel 'Panel 5' loaded with ID: 20
Panel 'Panel 6' loaded with ID: 21
Panel 'Panel 7' loaded with ID: 22
Panel 'Panel 8' loaded with ID: 23
Panel 'Panel 9' loaded with ID: 24
Panel 'Panel 10' loaded with ID: 25
Panel 'Panel 11' loaded with ID: 26
Panel 'Panel 12' loaded with ID: 27
Panel 'Panel 13' loaded with ID: 28
Panel 'Panel 14' loaded with ID: 29
Panel 'Panel 15' loaded with ID: 30
Panel 'Panel 16' loaded with ID: 31
Panel 'Panel 17' loaded with ID: 32
Panel 'Panel 18' loaded with ID: 33
Panel 'Panel 19' loaded with ID: 34
Panel 'Panel 20' loaded with ID: 35
Panel 'Panel 21' loaded with ID: 36
Panel 'Panel 22' loaded with ID: 37
Panel 'Panel 23' loaded with ID: 38
Panel 'Panel 24' loaded with ID: 39
Panel 'Panel 25' loaded with ID: 40
Panel 'Panel 26' loaded with ID: 41
Panel 'Panel 27' loaded with ID: 42
Panel 'Panel 28' loaded with ID: 43
P

,Sample name,Description,Stacking sequence,Laying up method,Curing process,Characterization methods,panel_id
0,Panel 1,Cured by Hexcel (last presentation),[+/0/-/90]2s,HLU,Vacuum bag,"C-scan, XCT, DSC",16
1,Panel 2,Laid up by Hexcel/ Cured at Imdea,[+/0/-/90]3s,HLU,Vacuum bag,"C-scan, XCT, DSC",17
2,Panel 3,Laid up by Hexcel/ Cured at Imdea,[+/0/-/90]3s,HLU,Hot press (7bar),"C-scan (cured), XCT (uncured)",18
3,Panel 4,Laid up by Hexcel/ Cured at Imdea,[+/0/-/90]3s,HLU,Vacuum bag,"C-scan, XCT",19
4,Panel 5,Laid up by Hexcel/,[+/0/-/90]3s,HLU,Vacuum bag (remove film),C-scan,20
5,Panel 6,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t1),"C-scan, XCT, DSC",21
6,Panel 7,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t2),"C-scan, XCT, DSC",22
7,Panel 8,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t3),"C-scan, XCT, DSC",23
8,Panel 9,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t4),"C-scan, XCT, DSC",24
9,Panel 10,Laid up by Hexcel,[+/0/-/90]3s,HLU,Vacuum bag (t4),"C-scan, XCT, DSC",25
